In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
from openpyxl.styles import Font
from scipy import stats

try:
    from google.colab import files as colab_files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# ------------------------------------------------------------------ 1. CARGA (lee directo del ZIP, sin descomprimir)
if EN_COLAB:
    print("Sube 'modelo_limpio.zip'")
    RUTA_ZIP = next(iter(colab_files.upload()))
else:
    RUTA_ZIP = os.environ.get("RUTA_ZIP", "modelo_limpio.zip")

with zipfile.ZipFile(RUTA_ZIP) as z:
    leer = lambda nombre, cols=None, **kw: pd.read_csv(z.open(f"{nombre}.csv"), usecols=cols, **kw)
    f = leer("fact_atenciones", parse_dates=["fecha_hora_ingreso", "fecha"])
    pac = leer("dim_pacientes", ["paciente_id", "edad"])
    hos = leer("dim_hospitales", ["hospital_id", "institucion"])
    ser = leer("dim_servicios", ["servicio_id", "servicio", "tipo_atencion"])
    dia = leer("dim_diagnosticos", ["diagnostico_id", "diagnostico", "curso"])

# Tabla plana de analisis: solo se traen las columnas necesarias para no generar columnas repetidas
d = (f.merge(pac, on="paciente_id").merge(hos, on="hospital_id")
      .merge(ser, on="servicio_id").merge(dia, on="diagnostico_id"))
assert len(d) == len(f), "Un merge duplico o perdio filas: revisa las llaves"

d = d.rename(columns={"institucion": "hospital"})
d["anio"] = d.fecha.dt.year
d["anio_mes"] = d.fecha.dt.to_period("M").astype(str)
# Espera "esperada" = promedio de TODA la red para ese triage. Asi, la espera ajustada de un hospital
# muestra cuanto se aleja de lo esperado segun su mezcla de pacientes.
d["espera_ajustada"] = d.tiempo_espera_min - d.groupby("triage").tiempo_espera_min.transform("mean")


# ------------------------------------------------------------------ 2. PRUEBAS (3 funciones reutilizables)
def kruskal(col, por, datos=None):
    """Diferencia de una variable numerica entre grupos (no asume distribucion normal)."""
    datos = d if datos is None else datos
    r = stats.kruskal(*[g.dropna() for _, g in datos.groupby(por)[col]])
    return "Kruskal-Wallis", r.statistic, r.pvalue


def chi2(por, flag, datos=None):
    """Relacion entre una categoria y un indicador 0/1 (reingreso, hospitalizacion...)."""
    datos = d if datos is None else datos
    est, p, *_ = stats.chi2_contingency(pd.crosstab(datos[por], datos[flag]))
    return "Chi-cuadrado", est, p


def spearman(a, b):
    """Correlacion entre dos variables numericas (rho de -1 a 1)."""
    r = stats.spearmanr(d[a], d[b], nan_policy="omit")
    return "Spearman", r.statistic, r.pvalue


def tendencia(col):
    """Pendiente de la serie mensual: hay tendencia sostenida o es ruido?"""
    m = d.groupby("anio_mes")[col].mean().to_numpy()
    r = stats.linregress(np.arange(len(m)), m)
    return "Regresion lineal mensual", r.slope, r.pvalue


# Comparacion interanual justa: mismos meses de ambos anios (evita comparar un anio completo vs uno parcial)
ult_mes = d.fecha.max().month
mismo_rango = d[d.fecha.dt.month <= ult_mes]
a1, a2 = sorted(mismo_rango.anio.unique())[:2]
espera_a = {a: mismo_rango.loc[mismo_rango.anio == a, "tiempo_espera_min"].dropna() for a in (a1, a2)}


def interanual_espera():
    r = stats.mannwhitneyu(espera_a[a1], espera_a[a2])
    return "Mann-Whitney", r.statistic, r.pvalue


# Valores para las lecturas
adj = d.groupby("hospital").espera_ajustada.mean()
mean_ = lambda col, por: d.groupby(por)[col].mean()
rate = lambda datos, c: datos[c].mean()

# (id, que se prueba, resultado de la prueba, lectura en palabras)
filas = [
    ("H1a", "Tendencia mensual de la espera", tendencia("tiempo_espera_min"),
     "Pendiente en min/mes: cerca de 0 y p alto = sin tendencia creciente"),
    ("H1b", "Tendencia mensual del reingreso", tendencia("reingreso_30_dias"),
     "Pendiente en proporcion/mes: cerca de 0 y p alto = sin tendencia creciente"),
    ("H1c", f"Espera meses 1-{ult_mes}: {a1} vs {a2}", interanual_espera(),
     f"Espera {espera_a[a1].mean():.1f} vs {espera_a[a2].mean():.1f} min"),
    ("H1d", f"Reingreso meses 1-{ult_mes}: {a1} vs {a2}", chi2("anio", "reingreso_30_dias", mismo_rango),
     f"Reingreso {rate(mismo_rango[mismo_rango.anio == a1], 'reingreso_30_dias'):.1%} vs "
     f"{rate(mismo_rango[mismo_rango.anio == a2], 'reingreso_30_dias'):.1%}"),
    ("H2", "Espera ajustada por triage entre hospitales", kruskal("espera_ajustada", "hospital"),
     f"{adj.idxmax()} {adj.max():+.1f} min; {adj.idxmin()} {adj.min():+.1f} min vs lo esperado por su triage"),
    ("H3", "Satisfaccion entre hospitales", kruskal("satisfaccion_1_a_5", "hospital"),
     f"Va de {mean_('satisfaccion_1_a_5', 'hospital').min():.2f} a {mean_('satisfaccion_1_a_5', 'hospital').max():.2f} (escala 1-5): significativo pero pequeno"),
    ("H4a", "Costo entre servicios", kruskal("costo_atencion_cop", "servicio"),
     f"Costo prom de ${mean_('costo_atencion_cop', 'servicio').min():,.0f} a ${mean_('costo_atencion_cop', 'servicio').max():,.0f}"),
    ("H4b", "Costo entre hospitales", kruskal("costo_atencion_cop", "hospital"),
     f"Costo prom de ${mean_('costo_atencion_cop', 'hospital').min():,.0f} a ${mean_('costo_atencion_cop', 'hospital').max():,.0f}"),
    ("H5a", "Reingreso segun diagnostico", chi2("diagnostico", "reingreso_30_dias"),
     f"Cronicos {d[d.curso == 'Cronica'].reingreso_30_dias.mean():.1%} vs resto {d[d.curso != 'Cronica'].reingreso_30_dias.mean():.1%}"),
    ("H5b", "Reingreso segun hospital", chi2("hospital", "reingreso_30_dias"),
     f"Va de {mean_('reingreso_30_dias', 'hospital').min():.1%} a {mean_('reingreso_30_dias', 'hospital').max():.1%}"),
    ("H6", "Hospitalizacion segun tipo de servicio", chi2("tipo_atencion", "es_hospitalizado"),
     " | ".join(f"{k}: {v:.1%}" for k, v in mean_("es_hospitalizado", "tipo_atencion").items())),
    ("R1", "Espera vs satisfaccion", spearman("tiempo_espera_min", "satisfaccion_1_a_5"),
     "rho negativo: a mas espera, menos satisfaccion (asociacion, no causalidad)"),
    ("R2", "Triage vs espera", spearman("triage", "tiempo_espera_min"),
     "rho positivo: triage menos urgente espera mas"),
    ("R4", "Estancia vs costo", spearman("estancia_dias", "costo_atencion_cop"),
     "rho alto: la estancia es el principal motor del costo"),
    ("R5", "Espera vs reingreso", spearman("tiempo_espera_min", "reingreso_30_dias"),
     "rho ~ 0: la espera no se relaciona con el reingreso"),
]

pruebas = pd.DataFrame(
    [{"id": i, "que_se_prueba": q, "prueba": r[0], "estadistico": r[1], "p_valor": max(r[2], 1e-300),
      "significativo_p<0.05": "Si" if r[2] < 0.05 else "No", "lectura": txt} for i, q, r, txt in filas])


# ------------------------------------------------------------------ 3. EXPORTAR
SALIDA = "Evidencia_Estadistica.xlsx"
with pd.ExcelWriter(SALIDA, engine="openpyxl") as w:
    pruebas.to_excel(w, sheet_name="Pruebas", index=False)
    for ws in w.book.worksheets:
        for c in ws[1]:
            c.font = Font(bold=True)
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(len(str(col[0].value)) + 4, 14), 70)
    for fila in w.book["Pruebas"].iter_rows(min_row=2):
        fila[3].number_format = "0.000"      # estadistico
        fila[4].number_format = "0.0E+00"    # p-valor en notacion cientifica

pd.set_option("display.width", 220, "display.max_colwidth", 70)
print(pruebas[["id", "que_se_prueba", "p_valor", "significativo_p<0.05", "lectura"]].to_string(index=False))
print(f"\nListo: {SALIDA}")
if EN_COLAB:
    colab_files.download(SALIDA)

Sube 'modelo_limpio.zip'


Saving modelo_limpio.zip to modelo_limpio (1).zip
 id                               que_se_prueba       p_valor significativo_p<0.05                                                                             lectura
H1a              Tendencia mensual de la espera  7.849832e-01                   No                 Pendiente en min/mes: cerca de 0 y p alto = sin tendencia creciente
H1b             Tendencia mensual del reingreso  4.082820e-01                   No          Pendiente en proporcion/mes: cerca de 0 y p alto = sin tendencia creciente
H1c             Espera meses 1-10: 2025 vs 2026  6.247862e-01                   No                                                           Espera 101.2 vs 102.3 min
H1d          Reingreso meses 1-10: 2025 vs 2026  1.841729e-01                   No                                                              Reingreso 7.7% vs 8.8%
 H2 Espera ajustada por triage entre hospitales  6.046812e-59                   Si Hospital Regional +9.8 min; Clin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>